In [0]:
%sql
USE CATALOG dados_fa;
USE SCHEMA pessoal;
CREATE VOLUME IF NOT EXISTS org_xml;

In [0]:
import os
import re
import json
import pandas as pd
import xmltodict

from lxml import etree
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

BASE_PATH = "/Volumes/dados_fa/pessoal/xml_files"
TARGET_TABLE = "dados_fa.pessoal.org_xml"
INVALID_XML_CHARS_RE = re.compile(
    r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]"
)
AMP_RE = re.compile(r"&(?!amp;|lt;|gt;|quot;|apos;|#\d+;|#x[0-9A-Fa-f]+;)")


schema_artigos = StructType([
    StructField("id", StringType(), True),
    StructField("idOficio", StringType(), True),
    StructField("pubName", StringType(), True),
    StructField("numberPage", StringType(), True),
    StructField("name", StringType(), True),
    StructField("artType", StringType(), True),
    StructField("artClass", StringType(), True),
    StructField("artSize", StringType(), True),
    StructField("artNotes", StringType(), True),
    StructField("pubDate", StringType(), True),
    StructField("artCategory", StringType(), True),
    StructField("pdfPage", StringType(), True),
    StructField("editionNumber", StringType(), True),
    StructField("highlightType", StringType(), True),
    StructField("highlightPriority", StringType(), True),
    StructField("highlight", StringType(), True),
    StructField("highlightimage", StringType(), True),
    StructField("highlightimagename", StringType(), True),
    StructField("idMateria", StringType(), True),
    StructField("identifica", StringType(), True),
    StructField("data", StringType(), True),
    StructField("ementa", StringType(), True),
    StructField("titulo", StringType(), True),
    StructField("subtitulo", StringType(), True),
    StructField("texto", StringType(), True),
    StructField("midia", StringType(), True),
    StructField("arquivo_origem", StringType(), True),
    StructField("error_parse", StringType(), True)
])

def clean_xml_text(text: str) -> str:
    text = INVALID_XML_CHARS_RE.sub("", text)
    return text

def fix_unescaped_ampersands(text: str) -> str:
    return AMP_RE.sub("&amp;", text)

def normalize_scalar(value):
    if value is None:
        return None
    if isinstance(value, str):
        return value
    if isinstance(value, (int, float, bool)):
        return str(value)
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def parse_xml_robusto(content: bytes):
    text = content.decode("utf-8", errors="ignore")
    text = clean_xml_text(text)
    text = fix_unescaped_ampersands(text)

    try:
        return xmltodict.parse(text), None
    except Exception as e1:
        try:
            parser = etree.XMLParser(recover=True)
            root = etree.fromstring(text.encode("utf-8"), parser=parser)
            repaired = etree.tostring(root, encoding="utf-8")
            return xmltodict.parse(repaired), f"recover_lxml: {e1}"
        except Exception as e2:
            return None, f"{e1} | fallback_lxml: {e2}"

def parse_xml_batches(iterator):
    for pdf in iterator:
        rows = []

        for _, row in pdf.iterrows():
            path = row["arquivo_origem"]
            content = row["content"]

            try:
                doc, parse_warning = parse_xml_robusto(content)

                if doc is None:
                    raise ValueError(parse_warning)

                xml_root = doc.get("xml") or {}
                art = xml_root.get("article") or {}
                body = art.get("body") or {}
                midias = art.get("Midias") or {}

                rows.append({
                    "id": normalize_scalar(art.get("@id")),
                    "idOficio": normalize_scalar(art.get("@idOficio")),
                    "pubName": normalize_scalar(art.get("@pubName")),
                    "numberPage": normalize_scalar(art.get("@numberPage")),
                    "name": normalize_scalar(art.get("@name")),
                    "artType": normalize_scalar(art.get("@artType")),
                    "artClass": normalize_scalar(art.get("@artClass")),
                    "artSize": normalize_scalar(art.get("@artSize")),
                    "artNotes": normalize_scalar(art.get("@artNotes")),
                    "pubDate": normalize_scalar(art.get("@pubDate")),
                    "artCategory": normalize_scalar(art.get("@artCategory")),
                    "pdfPage": normalize_scalar(art.get("@pdfPage")),
                    "editionNumber": normalize_scalar(art.get("@editionNumber")),
                    "highlightType": normalize_scalar(art.get("@highlightType")),
                    "highlightPriority": normalize_scalar(art.get("@highlightPriority")),
                    "highlight": normalize_scalar(art.get("@highlight")),
                    "highlightimage": normalize_scalar(art.get("@highlightimage")),
                    "highlightimagename": normalize_scalar(art.get("@highlightimagename")),
                    "idMateria": normalize_scalar(art.get("@idMateria")),
                    "identifica": normalize_scalar(body.get("Identifica")),
                    "data": normalize_scalar(body.get("Data")),
                    "ementa": normalize_scalar(body.get("Ementa")),
                    "titulo": normalize_scalar(body.get("Titulo")),
                    "subtitulo": normalize_scalar(body.get("SubTitulo")),
                    "texto": normalize_scalar(body.get("Texto")),
                    "midia": normalize_scalar(midias.get("Midia")),
                    "arquivo_origem": normalize_scalar(path),
                    "error_parse": normalize_scalar(parse_warning),
                })

            except Exception as e:
                rows.append({
                    "id": None,
                    "idOficio": None,
                    "pubName": None,
                    "numberPage": None,
                    "name": None,
                    "artType": None,
                    "artClass": None,
                    "artSize": None,
                    "artNotes": None,
                    "pubDate": None,
                    "artCategory": None,
                    "pdfPage": None,
                    "editionNumber": None,
                    "highlightType": None,
                    "highlightPriority": None,
                    "highlight": None,
                    "highlightimage": None,
                    "highlightimagename": None,
                    "idMateria": None,
                    "identifica": None,
                    "data": None,
                    "ementa": None,
                    "titulo": None,
                    "subtitulo": None,
                    "texto": None,
                    "midia": None,
                    "arquivo_origem": normalize_scalar(path),
                    "error_parse": normalize_scalar(str(e)),
                })

        if rows:
            yield pd.DataFrame(rows)


def processar_edicao(edicao_path: str):
    files_df = (
        spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", "*.xml")
        .load(edicao_path)
        .select(
            F.col("path").alias("arquivo_origem"),
            F.col("content")
        )
    )

    if spark.catalog.tableExists(TARGET_TABLE):
        existing_df = spark.table(TARGET_TABLE).select("arquivo_origem").distinct()
        pending_df = files_df.join(existing_df, on="arquivo_origem", how="left_anti")
    else:
        pending_df = files_df

    if pending_df.limit(1).count() == 0:
        print(f"Sem pendências em {edicao_path}")
        return 0

    parsed_df = (
        pending_df
        .repartition(32)
        .mapInPandas(parse_xml_batches, schema=schema_artigos)
    )

    (
        parsed_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET_TABLE)
    )

    qtd = parsed_df.count()
    print(f"{edicao_path}: {qtd} registros gravados")
    return qtd


edicoes = sorted(
    os.path.join(BASE_PATH, d)
    for d in os.listdir(BASE_PATH)
    if os.path.isdir(os.path.join(BASE_PATH, d))
)

total = 0
for edicao_path in edicoes:
    try:
        total += processar_edicao(edicao_path)
    except Exception as e:
        print(f"Erro em {edicao_path}: {e}")

print(f"Total gravado: {total}")